In [ ]:
import os
import shutil

# Remove existing directory if it exists
ROOT = '/kaggle/working/mini-gpt'
if os.path.exists(ROOT):
    shutil.rmtree(ROOT)

# Clone the repository from GitHub
!git clone https://github.com/maariogutierrez/mini-gpt.git {ROOT}
os.chdir(ROOT)

REQUIREMENTS = 'https://raw.githubusercontent.com/maariogutierrez/mini-gpt/main/requirements.txt'
!pip install -r $REQUIREMENTS

print("Logging into Weights & Biases (wandb). Follow the prompts.")
import wandb
wandb.login(key='')

# Store outputs in Google Drive, but keep code in cloned repo
OUTPUT_ROOT = '/kaggle/working/output'
os.makedirs(OUTPUT_ROOT, exist_ok=True)

DATA_DIR = os.path.join(OUTPUT_ROOT, 'data')
EXPORTS_DIR = os.path.join(OUTPUT_ROOT, 'exports')
LOGS_DIR = os.path.join(OUTPUT_ROOT, 'logs')
CHECKPOINTS_DIR = os.path.join(OUTPUT_ROOT, 'checkpoints')

for d in [DATA_DIR, EXPORTS_DIR, LOGS_DIR, CHECKPOINTS_DIR]:
    os.makedirs(d, exist_ok=True)

print("\n--- Kaggle Setup Summary ---")
print(f"Code repository: {ROOT}")
print(f"Output directory: {OUTPUT_ROOT}")
print(f"Current Working Directory: {os.getcwd()}")

Cloning into '/kaggle/working/mini-gpt'...
remote: Enumerating objects: 195, done.
remote: Counting objects: 100% (195/195), done.
remote: Compressing objects: 100% (138/138), done.
remote: Total 195 (delta 104), reused 143 (delta 52), pack-reused 0 (from 0)
Receiving objects: 100% (195/195), 1.30 MiB | 19.04 MiB/s, done.
Resolving deltas: 100% (104/104), done.
Logging into Weights & Biases (wandb). Follow the prompts.


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mariogutierrez (mariogutierrez-universidad-polit-cnica-de-madrid) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



--- Kaggle Setup Summary ---
Code repository: /kaggle/working/mini-gpt
Output directory: /kaggle/working/output
Current Working Directory: /kaggle/working/mini-gpt


In [ ]:
import gdown
import os

# Replace these IDs with your actual File IDs from Google Drive
files_to_download = {
    'train.bin': '',
    'val.bin': '',
    'checkpoint_step_1000.pt': ''
}

# Download the training data
print("Downloading train.bin...")
gdown.download(f'https://drive.google.com/uc?id={files_to_download["train.bin"]}', 
               '/kaggle/working/output/data/train.bin', quiet=False)

print("Downloading val.bin...")
gdown.download(f'https://drive.google.com/uc?id={files_to_download["val.bin"]}', 
               '/kaggle/working/output/data/val.bin', quiet=False)

# # Download the checkpoint
# print("Downloading checkpoint...")
# gdown.download(f'https://drive.google.com/uc?id={files_to_download["checkpoint_step_1000.pt"]}', 
#                '/kaggle/working/output/checkpoints/checkpoint_step_1000.pt', quiet=False)

print("Files successfully moved to Kaggle!")

Downloading...
From (original): https://drive.google.com/uc?id=1wRP5jrk6csXbvFLDJz1YtEAZLKZbQc0r
From (redirected): https://drive.google.com/uc?id=1wRP5jrk6csXbvFLDJz1YtEAZLKZbQc0r&confirm=t&uuid=2ebda4d7-611d-44e2-8356-720bc1b7e080
To: /kaggle/working/output/data/train.bin
100%|██████████| 853M/853M [00:07<00:00, 120MB/s]  


Downloading...
From: https://drive.google.com/uc?id=1QZNte5AFwJZ9MrmcOUu8LCKQR6e40MMI
To: /kaggle/working/output/data/val.bin
100%|██████████| 94.9M/94.9M [00:01<00:00, 50.3MB/s]

Files successfully moved to Kaggle!


In [ ]:

from model.architecture.gpt import GPT, GPTConfig
from model.training.trainer import Trainer
from model.training.dataset import TokenDataset
import torch


# Model configuration - balanced for efficient training
model_config = GPTConfig(
    vocab_size=50304,      
    block_size=256,        # Context window
    n_layer=12,            # 12 transformer layers
    n_head=12,             # 12 attention heads
    n_embd=768,            # 768 embedding dimension
    dropout=0.1
)

print("Creating model...")
model = GPT(model_config)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Device: {device}")

local_train_bin = os.path.join(DATA_DIR, 'train.bin')
local_val_bin = os.path.join(DATA_DIR, 'val.bin')

print("Loading datasets...")
train_dataset = TokenDataset(local_train_bin, model_config.block_size)
val_dataset = TokenDataset(local_val_bin, model_config.block_size)
print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset: {len(val_dataset)} samples")

trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    batch_size=16,
    accum_steps=32,        # 16 × 32 = 512 effective batch size
    learning_rate=6e-4,    # Moderate learning rate
    weight_decay=0.1,
    warmup_steps=1200,      # 1200 warmup steps
    max_steps=60000,        # Train for 60000 steps
    grad_clip=1.0,
    device=device,
    checkpoint_dir=CHECKPOINTS_DIR,
    wandb_project="mini-gpt",
    use_mixed_precision=True,
)

print(f"\nTraining configuration:")
print(f"  Effective batch size: {trainer.effective_batch_size}")
print(f"  Max steps: {trainer.max_steps}")

Creating model...
GPT Model Information:
  Total parameters: 123,886,080
  Trainable parameters: 123,886,080
  Config:
    - vocab_size: 50304
    - block_size: 256
    - n_layer: 12
    - n_head: 12
    - n_embd: 768
    - dropout: 0.1
Model parameters: 123,886,080
Device: cuda
Loading datasets...
Train dataset: 426536394 samples
Val dataset: 47455100 samples

Training configuration:
  Effective batch size: 512
  Max steps: 60000


In [ ]:

# Run training
print("=" * 60)
print("Starting training session")
print("=" * 60)

# checkpoint_path = os.path.join(CHECKPOINTS_DIR, "checkpoint_step_1000.pt")
# trainer.load_checkpoint(checkpoint_path)
trainer.train()

print("\n" + "=" * 60)
print("Training session complete!")
print("=" * 60)
print(f"Final step: {trainer.global_step}")
print(f"Best validation loss: {trainer.best_val_loss:.4f}")


Starting training session


/kaggle/working/mini-gpt/model/training/dataset.py:15: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  x = torch.from_numpy(self.tokens[idx : idx + self.block_size]).long()


Step 100 | Loss: 6.3246 | LR: 5.00e-05
Step 200 | Loss: 4.0806 | LR: 1.00e-04
Step 300 | Loss: 3.4535 | LR: 1.50e-04


In [ ]:
import os
import torch

from model.architecture.gpt import GPT, GPTConfig
from model.architecture.tokenizer import CustomTokenizer

tokenizer = CustomTokenizer()

model_config = GPTConfig(
    vocab_size=50304,      
    block_size=512,        # Context window
    n_layer=12,             # 8 transformer layers
    n_head=12,              # 8 attention heads
    n_embd=768,            # 512 embedding dimension
    dropout=0.1
)

device = "cuda" if torch.cuda.is_available() else "cpu"

# Recreate the model architecture
model = GPT(model_config).to(device)

# Load best checkpoint weights
best_checkpoint_path = os.path.join(CHECKPOINTS_DIR, "best_model.pt")
checkpoint = torch.load(best_checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print(f"Loaded best model from: {best_checkpoint_path}")
print(f"Best validation loss: {checkpoint['best_val_loss']:.4f}")

# Generate text
prompt = "Once upon a time"
idx = torch.tensor([tokenizer.encode(prompt)], device=device)

generated = model.generate(idx, max_new_tokens=100, temperature=0.8)
text = tokenizer.decode(generated[0].tolist())

print(text)